In [ ]:
# Instalando bibliotecas
!pip3 install pytorch-lightning torchinfo torchmetrics torch-summary seaborn scikit-learn

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try 'pacman -S
    python-xyz', where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Arch-packaged Python package,
    create a virtual environment using 'python -m venv path/to/venv'.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip.
    
    If you wish to install a non-Arch packaged Python application,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. Make sure you have python-pipx
    installed via pacman.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detailed specification.


In [4]:
# Importando as bibliotecas que serão utilizadas
import multiprocessing as mp
import os
import random
import sys
from glob import glob

import matplotlib.pyplot as plt
import numpy as np
import pytorch_lightning as pl
import torch
import torchinfo
from pytorch_lightning.callbacks import ModelCheckpoint
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset
from torchmetrics import Accuracy
from torchsummary import summary
from torchvision import datasets, models
from torchvision.transforms import ToTensor
from torchvision.utils import make_grid

import seaborn as sns
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import torch.optim as optim
from torch.utils.data import Subset
from torchvision import transforms
from PIL import Image

%load_ext tensorboard

ModuleNotFoundError: No module named 'pytorch_lightning'

In [ ]:
# Configurações iniciais

DATASET_PATH = "photos"
BATCH_SIZE = 32
IMG_SIZE = 224  # Tamanho para resize (modelos pré-treinados usam 224)
NUM_EPOCHS = 20
LEARNING_RATE = 0.001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")

In [ ]:
# Carregar e pré-processar o dataset

# Transformações para treino com augmentation
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Transformações para validação (sem augmentation)
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Carrega dataset completo
full_dataset = datasets.ImageFolder(root=DATASET_PATH, transform=train_transform)

# Mapeamento de índices para nomes das classes
class_names = full_dataset.classes
num_classes = len(class_names)
print(f"Classes encontradas: {class_names}")

In [ ]:
# Dividir em treino e validação

indices = list(range(len(full_dataset)))
train_indices, val_indices = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=full_dataset.targets
)

# Datasets separados
train_dataset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, val_indices)

val_dataset.dataset.transform = val_transform

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
# Definir a CNN
class CNNModel(nn.Module):
    def __init__(self, num_classes):
        super(CNNModel, self).__init__()
        # Usando ResNet18 pré-treinada
        self.base_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        # Congelar camadas iniciais (opcional)
        for param in self.base_model.parameters():
            param.requires_grad = False
        # Substituir a última camada fully connected
        in_features = self.base_model.fc.in_features
        self.base_model.fc = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.base_model(x)

model = CNNModel(num_classes).to(DEVICE)

#### Talvez não incluir
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
def train_one_epoch(loader, model, optimizer, criterion):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [ ]:
def validate(loader, model, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc, all_preds, all_labels

In [ ]:
# Loop de treinamento
train_losses, train_accs = [], []
val_losses, val_accs = [], []

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(train_loader, model, optimizer, criterion)
    val_loss, val_acc, _, _ = validate(val_loader, model, criterion)

    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

In [ ]:
# Avaliação final
val_loss, val_acc, all_preds, all_labels = validate(val_loader, model, criterion)
print(f"\nAcurácia final na validação: {val_acc:.4f}")

In [ ]:
# Análises quantitativas e qualitativas
# Curvas de aprendizado
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(train_losses, label='Treino')
plt.plot(val_losses, label='Validação')
plt.title('Perda durante treinamento')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1,2,2)
plt.plot(train_accs, label='Treino')
plt.plot(val_accs, label='Validação')
plt.title('Acurácia durante treinamento')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.legend()
plt.tight_layout()
plt.savefig('curvas_treinamento.png')
plt.show()

In [ ]:
# Matriz de confusão
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Matriz de Confusão')
plt.xlabel('Predito')
plt.ylabel('Verdadeiro')
plt.savefig('matriz_confusao.png')
plt.show()

In [ ]:
# Relatório de classificação
print("\nRelatório de Classificação:")
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
# Análise qualitativa: visualizar exemplos classificados corretamente e incorretamente
def imshow(inp, title=None):
    """Função para mostrar imagem (desnormalizada)"""
    inp = inp.cpu().numpy().transpose((1,2,0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    plt.imshow(inp)
    if title:
        plt.title(title)
    plt.axis('off')

# Obter algumas imagens do val_loader
model.eval()
images, labels = next(iter(val_loader))
images, labels = images.to(DEVICE), labels.to(DEVICE)
outputs = model(images)
_, preds = torch.max(outputs, 1)

# Mostrar algumas imagens com seus rótulos reais e preditos
plt.figure(figsize=(15, 8))
for i in range(min(8, len(images))):
    ax = plt.subplot(2, 4, i+1)
    title = f"Real: {class_names[labels[i]]}\nPred: {class_names[preds[i]]}"
    imshow(images[i], title)
    if labels[i] != preds[i]:
        ax.title.set_color('red')
plt.tight_layout()
plt.savefig('exemplos_classificacao.png')
plt.show()

In [ ]:
# Identificar padrões de erro
errors_idx = [i for i, (true, pred) in enumerate(zip(all_labels, all_preds)) if true != pred]
print(f"\nTotal de erros na validação: {len(errors_idx)} de {len(all_labels)} imagens")
print("Exemplos de erros mais frequentes:")
error_pairs = [(all_labels[i], all_preds[i]) for i in errors_idx]
from collections import Counter
for (true, pred), count in Counter(error_pairs).most_common(5):
    print(f"  {class_names[true]} classificado como {class_names[pred]}: {count} vezes")